<a href="https://colab.research.google.com/github/prometricas/William_Rondon/blob/main/William_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Análisis estadístico del instrumento COCTS**

Este cuaderno tiene cuatro objetivos principales:

1. **Reconstruir** los puntajes normalizados del COCTS para el **piloto** a partir del libro de Excel.
2. **Construir** un `DataFrame` limpio y analítico.
3. **Estimar** confiabilidad global, por dimensión y por ítem.
4. **Visualizar** los resultados con gráficos comparativos y estéticos.

---

## Productos esperados

| Bloque | Salida principal |
|---|---|
| Extracción | `df_piloto` |
| Metadatos | `meta_items` |
| Confiabilidad | `tabla_confiabilidad`, `tabla_items` |
| Visualización | histogramas, boxplots, violin plots, heatmap |
| Comparación futura | función `comparar_pre_post()` |

---

## Lógica de puntuación que voy a replicar

Para cada frase del COCTS, convierto la respuesta Likert \(r \in \{1,\dots,9\}\) a un índice normalizado según la categoría de la frase:

- **A** = Adecuada  
- **P** = Plausible  
- **I** = Ingenua

Luego, para cada cuestión **_j_** y estudiante **_i_**, calculo el índice ponderado de la cuestión como el promedio de los promedios por categoría presentes en esa cuestión:

$$
I^{(\text{pond})}_{ij}
=
\frac{
\overline{I^{(A)}_{ij}} + \overline{I^{(P)}_{ij}} + \overline{I^{(I)}_{ij}}
}{m_{ij}}
$$

donde $m_{ij}$ es el número de categorías presentes para esa cuestión.

Después calculo:

### Puntaje por dimensión
$$
D_{id} = \frac{1}{k_d} \sum_{j \in d} I^{(\text{pond})}_{ij}
$$

### Puntaje global
$$
G_i = \frac{1}{15} \sum_{j=1}^{15} I^{(\text{pond})}_{ij}
$$

---

## Nota metodológica

- En el **piloto**, el libro no trae nombres ni variables demográficas dentro de la hoja de captura principal.
- Por eso crearé las columnas `nombre`, `sexo` y `edad` vacías.
- Si luego existe un roster auxiliar del piloto, lo uniré por `codigo`.

# **1. Crear dataset del instrumento**

En esta sección:

- subo el archivo de Excel a Colab,
- verifico las hojas disponibles,
- declaro el nombre de la hoja piloto,
- y fijo la estructura conceptual del COCTS:

| Dimensión | Ítems |
|---|---|
| Definiciones de ciencia y tecnología | 10111, 10411 |
| Sociología externa | 20141, 20511, 40211, 40451 |
| Sociología interna | 60111, 60611, 70231, 70721 |
| Epistemología | 90211, 90311, 90411, 90651, 91011 |

También dejo explícita la métrica de transformación de las respuestas Likert a índices normalizados.

In [15]:
# Limpiar memoria
from IPython import get_ipython

# Borrar todas las variables en memoria
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('reset', '-f')

# Re-importar módulos después de limpiar la memoria
import os
import shutil

# Borrar todos los archivos y subdirectorios en /content/ (excepto sample_data)
for item in os.listdir('/content/'):
    if item == 'sample_data':
        continue
    item_path = os.path.join('/content/', item)
    if os.path.isfile(item_path):
        os.remove(item_path)
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)

print("Variables en memoria borradas y archivos en '/content/' (excepto 'sample_data') eliminados.")

Variables en memoria borradas y archivos en '/content/' (excepto 'sample_data') eliminados.


In [16]:
import pandas as pd
from google.colab import files

# =========================
# 1) Subir archivo
# =========================
uploaded = files.upload()
file_path = next(iter(uploaded))   # nombre del archivo subido

# =========================
# 2) Leer hojas
# =========================
piloto_raw = pd.read_excel(file_path, sheet_name="Piloto 1101", header=None)
est_raw = pd.read_excel(file_path, sheet_name="Estudiantes", header=None)

# =========================
# 3) Tomar puntajes YA NORMALIZADOS del piloto
#    En "Piloto 1101" hay dos bloques de Est 1..Est n:
#    - primero: respuestas crudas
#    - segundo: puntajes normalizados
# =========================
header = piloto_raw.iloc[2]  # fila donde están los encabezados
est_cols = [i for i, v in enumerate(header) if isinstance(v, str) and v.startswith("Est ")]

n_est = len(est_cols) // 2
norm_cols = est_cols[n_est:]                 # segundo bloque = normalizados
student_codes = [header[i] for i in norm_cols]

# Filas "índice pond" = puntaje final por ítem
item_codes = piloto_raw.iloc[:, 0].ffill()

mask_pond = (
    piloto_raw.iloc[:, 2]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("índice pond")
)

puntajes = piloto_raw.loc[mask_pond, norm_cols].copy()
puntajes.index = item_codes[mask_pond].astype(int).map(lambda x: f"q_{x}")
puntajes.columns = student_codes

# Transponer: una fila por estudiante
df_puntajes = puntajes.T.reset_index().rename(columns={"index": "codigo"})

# =========================
# 4) Tomar demografía del grupo CONTROL
#    En la hoja "Estudiantes", CONTROL está a la derecha:
#    col 5 = Código
#    col 6 = Nombre
#    col 7 = Sexo
#    col 8 = Edad
# =========================
roster_control = est_raw.iloc[2:, [5, 6, 7, 8]].copy()
roster_control.columns = ["codigo", "nombre", "sexo", "edad"]

roster_control = roster_control[
    roster_control["codigo"].astype(str).str.startswith("Est ", na=False)
].copy()

# =========================
# 5) Unir demografía + puntajes
# =========================
df_cocts_piloto = roster_control.merge(df_puntajes, on="codigo", how="left")

# Ordenar columnas
item_cols = [c for c in df_cocts_piloto.columns if str(c).startswith("q_")]
df_cocts_piloto = df_cocts_piloto[["codigo", "nombre", "sexo", "edad"] + item_cols]

# Tipos
df_cocts_piloto["edad"] = pd.to_numeric(df_cocts_piloto["edad"], errors="coerce")
for c in item_cols:
    df_cocts_piloto[c] = pd.to_numeric(df_cocts_piloto[c], errors="coerce")

# =========================
# 6) Resultado final
# =========================
print("Dimensión:", df_cocts_piloto.shape)
display(df_cocts_piloto.head())

Saving COCTS Piloto y Pre Test.xlsx to COCTS Piloto y Pre Test.xlsx
Dimensión: (28, 19)


,codigo,nombre,sexo,edad,q_10111,q_10411,q_20141,q_20511,q_40211,q_40451,q_60111,q_60611,q_70231,q_70721,q_90211,q_90311,q_90411,q_90651,q_91011
0,Est 1,Hanna Guarumo,F,16,0.458333,0.166667,0.229167,0.097222,0.388889,0.041667,0.347222,0.000000,0.333333,0.125000,-0.263889,0.416667,0.333333,0.138889,-0.277778
1,Est 2,Ana Mileth Collo Valencia,F,16,-0.125000,-0.416667,0.333333,0.222222,-0.916667,0.708333,-0.375000,-0.069444,0.145833,-0.430556,-0.222222,0.166667,0.083333,0.208333,0.500000
2,Est 3,Pineda Caro Liz,F,15,-0.333333,0.083333,-0.083333,-0.319444,-0.791667,0.347222,0.583333,-0.277778,0.208333,-0.277778,-0.166667,-0.083333,0.125000,-0.208333,0.000000
3,Est 4,Anny Jireth Cabrera Calderon,F,17,0.166667,0.000000,-0.041667,-0.027778,0.055556,0.513889,0.097222,0.111111,0.229167,0.180556,0.472222,0.277778,0.375000,0.000000,0.277778
4,Est 5,Isabella Victorino Nuñez,F,15,-0.050000,-0.125000,0.562500,0.347222,0.083333,0.388889,-0.069444,-0.104167,0.187500,0.055556,-0.125000,0.055556,0.250000,-0.305556,-0.444444


# **2. Confiabilidad del COCTS piloto**

En esta sección se evalúa la **consistencia interna** del instrumento COCTS usando los **puntajes normalizados por ítem** ya extraídos desde el archivo Excel.

El COCTS, en esta investigación, quedó conformado por **15 ítems** distribuidos en **4 dimensiones**:

| Dimensión | Ítems |
|---|---|
| Definiciones de ciencia y tecnología | 10111, 10411 |
| Sociología externa | 20141, 20511, 40211, 40451 |
| Sociología interna | 60111, 60611, 70231, 70721 |
| Epistemología | 90211, 90311, 90411, 90651, 91011 |

Además, el instrumento parte de frases clasificadas como **Adecuadas (A)**, **Plausibles (P)** o **Ingenuas (I)**, y sus respuestas son transformadas a un **índice normalizado** dentro del intervalo:

$$
-1 \le x \le 1
$$

Eso implica que, para esta etapa, los ítems del `dataframe` se comportan como **variables cuantitativas acotadas**, ya procesadas y comparables entre sí.

---

## ¿Qué tipos de confiabilidad se calcularán?

### 1. Alfa de Cronbach
Mide la consistencia interna de una escala a partir de la covariación entre ítems.

$$
\alpha = \frac{k}{k-1}\left(1 - \frac{\sum_{i=1}^{k} s_i^2}{s_T^2}\right)
$$

donde:

- $k$ = número de ítems
- $s_i^2$ = varianza del ítem $i$
- $s_T^2$ = varianza del puntaje total de la escala

Se calculará para:

- la escala total
- cada dimensión

---

### 2. Omega de McDonald
Es un estimador de consistencia interna menos restrictivo que el alfa, porque no exige que todos los ítems aporten exactamente la misma cantidad al constructo.

Si $\lambda_i$ representa la carga factorial del ítem $i$ en un modelo unifactorial, y $\theta_i$ su varianza de error, entonces:

$$
\omega = \frac{(\sum \lambda_i)^2}{(\sum \lambda_i)^2 + \sum \theta_i}
$$

Se usará como complemento del alfa para la escala total y para las dimensiones con suficiente número de ítems.

---

### 3. Confiabilidad por mitades (Split-half)
Se divide la escala en dos mitades y se correlacionan sus puntuaciones. Luego se corrige con la fórmula de Spearman-Brown:

$$
r_{SB} = \frac{2r}{1+r}
$$

donde $r$ es la correlación entre ambas mitades.

Esto aporta una segunda evidencia de consistencia interna para la escala total.

---

### 4. Correlación ítem-total corregida
Para cada ítem se calcula su correlación con el total de la escala **sin incluir ese mismo ítem**:

$$
r_{it(corr)} = \text{corr}(X_i, T - X_i)
$$

Sirve para identificar qué tan bien se integra cada ítem con el resto de la escala.

---

### 5. Alfa si se elimina el ítem
Se recalcula el alfa de Cronbach excluyendo cada ítem, uno por uno.

Esto permite evaluar si algún ítem está deteriorando la consistencia interna del instrumento.

---

### 6. Coeficiente Spearman-Brown para la dimensión de 2 ítems
La dimensión **Definiciones de ciencia y tecnología** solo tiene **2 ítems**, por lo que además del alfa conviene reportar la correlación entre ítems y su corrección mediante Spearman-Brown:

$$
r_{SB} = \frac{2r_{12}}{1+r_{12}}
$$

En escalas de 2 ítems, este índice suele ser más interpretable que el alfa.

---

## Criterio práctico de interpretación

| Indicador | Lectura práctica |
|---|---|
| $\alpha \ge 0.70$ | consistencia interna aceptable |
| $\alpha \ge 0.80$ | buena |
| $\alpha \ge 0.90$ | muy alta |
| $\omega$ | complemento más robusto que el alfa |
| $r_{it(corr)} < 0.20$ | ítem débil o poco alineado |
| alfa si se elimina | si sube claramente, el ítem merece revisión |
| Spearman-Brown | especialmente útil en subescalas muy cortas |

---

## Alcance de esta sección

Aquí se analizará:

1. **Confiabilidad global** del COCTS piloto.
2. **Confiabilidad por dimensión**.
3. **Diagnóstico por ítem**.

La variable de entrada será el `DataFrame` ya construido:

$$
\texttt{df\_cocts\_piloto}
$$

y las columnas de los ítems deben tener la forma:

$$
\texttt{q\_10111,\ q\_10411,\ \dots,\ q\_91011}
$$

In [17]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1) Definir columnas de ítems y dimensiones
# =========================================================
dimensiones = {
    "Definiciones de ciencia y tecnología": ["q_10111", "q_10411"],
    "Sociología externa": ["q_20141", "q_20511", "q_40211", "q_40451"],
    "Sociología interna": ["q_60111", "q_60611", "q_70231", "q_70721"],
    "Epistemología": ["q_90211", "q_90311", "q_90411", "q_90651", "q_91011"],
}

item_cols = [col for cols in dimensiones.values() for col in cols]

# Validación rápida
faltantes = [c for c in item_cols if c not in df_cocts_piloto.columns]
if faltantes:
    raise ValueError(f"Faltan estas columnas en df_cocts_piloto: {faltantes}")

df_items = df_cocts_piloto[item_cols].copy().apply(pd.to_numeric, errors="coerce")

# =========================================================
# 2) Funciones de confiabilidad
# =========================================================
def cronbach_alpha(df):
    X = df.dropna(axis=0, how="any")
    k = X.shape[1]
    if k < 2 or X.shape[0] < 2:
        return np.nan
    item_vars = X.var(axis=0, ddof=1)
    total_var = X.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return np.nan
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

def omega_mcdonald(df):
    """
    Omega total aproximado mediante análisis factorial de 1 factor.
    """
    X = df.dropna(axis=0, how="any")
    if X.shape[1] < 3 or X.shape[0] < 5:
        return np.nan

    Z = StandardScaler().fit_transform(X)
    fa = FactorAnalysis(n_components=1, random_state=123)
    fa.fit(Z)

    loadings = fa.components_.flatten()
    uniqueness = fa.noise_variance_

    num = (np.sum(loadings)) ** 2
    den = num + np.sum(uniqueness)

    return num / den if den != 0 else np.nan

def split_half_sb(df):
    """
    Confiabilidad por mitades con corrección de Spearman-Brown.
    Divide ítems en impares y pares.
    """
    X = df.dropna(axis=0, how="any")
    k = X.shape[1]
    if k < 2 or X.shape[0] < 3:
        return np.nan, np.nan

    cols1 = X.columns[::2]
    cols2 = X.columns[1::2]

    # Si una mitad queda vacía, no se puede calcular
    if len(cols1) == 0 or len(cols2) == 0:
        return np.nan, np.nan

    score1 = X[cols1].mean(axis=1)
    score2 = X[cols2].mean(axis=1)

    r = score1.corr(score2)
    sb = (2 * r) / (1 + r) if pd.notna(r) and (1 + r) != 0 else np.nan
    return r, sb

def item_total_correlations(df):
    X = df.dropna(axis=0, how="any")
    out = []

    for col in X.columns:
        total_without_item = X.drop(columns=[col]).sum(axis=1)
        rit = X[col].corr(total_without_item)
        alpha_deleted = cronbach_alpha(X.drop(columns=[col]))
        out.append({
            "item": col,
            "rit_corregida": rit,
            "alpha_si_se_elimina": alpha_deleted
        })

    return pd.DataFrame(out)

def spearman_brown_2items(df):
    X = df.dropna(axis=0, how="any")
    if X.shape[1] != 2 or X.shape[0] < 3:
        return np.nan, np.nan
    r = X.iloc[:, 0].corr(X.iloc[:, 1])
    sb = (2 * r) / (1 + r) if pd.notna(r) and (1 + r) != 0 else np.nan
    return r, sb

# =========================================================
# 3) Confiabilidad global
# =========================================================
alpha_total = cronbach_alpha(df_items)
omega_total = omega_mcdonald(df_items)
r_split, sb_total = split_half_sb(df_items)

resumen_global = pd.DataFrame([{
    "Escala": "COCTS total",
    "N_casos": df_items.dropna().shape[0],
    "N_items": df_items.shape[1],
    "Alpha_Cronbach": alpha_total,
    "Omega_McDonald": omega_total,
    "r_split_half": r_split,
    "Spearman_Brown_split_half": sb_total
}])

# =========================================================
# 4) Confiabilidad por dimensión
# =========================================================
filas_dim = []

for nombre_dim, cols in dimensiones.items():
    sub = df_cocts_piloto[cols].copy().apply(pd.to_numeric, errors="coerce")
    alpha_dim = cronbach_alpha(sub)
    omega_dim = omega_mcdonald(sub) if len(cols) >= 3 else np.nan

    fila = {
        "Dimensión": nombre_dim,
        "N_casos": sub.dropna().shape[0],
        "N_items": len(cols),
        "Alpha_Cronbach": alpha_dim,
        "Omega_McDonald": omega_dim,
        "r_entre_items": np.nan,
        "Spearman_Brown_2items": np.nan
    }

    if len(cols) == 2:
        r2, sb2 = spearman_brown_2items(sub)
        fila["r_entre_items"] = r2
        fila["Spearman_Brown_2items"] = sb2

    filas_dim.append(fila)

resumen_dimensiones = pd.DataFrame(filas_dim)

# =========================================================
# 5) Diagnóstico por ítem
# =========================================================
diagnostico_items = item_total_correlations(df_items)

# =========================================================
# 6) Redondear y mostrar
# =========================================================
resumen_global = resumen_global.round(4)
resumen_dimensiones = resumen_dimensiones.round(4)
diagnostico_items = diagnostico_items.round(4)

print("=== Confiabilidad global del COCTS piloto ===")
display(resumen_global)

print("\n=== Confiabilidad por dimensión ===")
display(resumen_dimensiones)

print("\n=== Diagnóstico por ítem ===")
display(diagnostico_items.sort_values("rit_corregida", ascending=False))

=== Confiabilidad global del COCTS piloto ===


,Escala,N_casos,N_items,Alpha_Cronbach,Omega_McDonald,r_split_half,Spearman_Brown_split_half
0,COCTS total,28,15,0.6025,0.6218,0.3507,0.5192



=== Confiabilidad por dimensión ===


,Dimensión,N_casos,N_items,Alpha_Cronbach,Omega_McDonald,r_entre_items,Spearman_Brown_2items
0,Definiciones de ciencia y tecnología,28,2,0.0917,NaN,0.0483,0.0922
1,Sociología externa,28,4,0.2510,0.4227,NaN,NaN
2,Sociología interna,28,4,0.1776,0.2503,NaN,NaN
3,Epistemología,28,5,0.4751,0.5184,NaN,NaN



=== Diagnóstico por ítem ===


,item,rit_corregida,alpha_si_se_elimina
0,q_10111,0.5781,0.5233
9,q_70721,0.5527,0.5419
3,q_20511,0.4420,0.5501
4,q_40211,0.3462,0.5620
11,q_90311,0.3183,0.5725
13,q_90651,0.2752,0.5806
10,q_90211,0.2411,0.5846
6,q_60111,0.2350,0.5853
14,q_91011,0.1803,0.5949
8,q_70231,0.1793,0.5961
